### Lab Exercise 4
### Implement Backpropagation in MLP. Predict the age of abalone from physical measurements.

### Dataset:
- Link 1: [UCI Machine Learning Repository] https://archive.ics.uci.edu/dataset/1/abalone
- Link 2: [Kaggle] https://www.kaggle.com/datasets/rodolfomendes/abalone-dataset


In [12]:
import pandas as pd
import numpy as np
df=pd.read_csv('/content/abalone.csv')
df.head()

,Sex,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4177 entries, 0 to 4176
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Sex             4177 non-null   object 
 1   Length          4177 non-null   float64
 2   Diameter        4177 non-null   float64
 3   Height          4177 non-null   float64
 4   Whole weight    4177 non-null   float64
 5   Shucked weight  4177 non-null   float64
 6   Viscera weight  4177 non-null   float64
 7   Shell weight    4177 non-null   float64
 8   Rings           4177 non-null   int64  
dtypes: float64(7), int64(1), object(1)
memory usage: 293.8+ KB


In [14]:
df["Sex_M"] = (df["Sex"] == "M").astype(int)
df["Sex_F"] = (df["Sex"] == "F").astype(int)
df["Sex_I"] = (df["Sex"] == "I").astype(int)
df.drop("Sex", axis=1, inplace=True)

In [15]:
# Features & Target
X = df.drop("Rings", axis=1).values
y = df["Rings"].values.reshape(-1, 1)

# Normalize
X = (X - X.mean(axis=0)) / X.std(axis=0)

# Train-Test Split
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Network Architecture
input_size = X_train.shape[1]
hidden_size = 10
output_size = 1

np.random.seed(42)
W1 = np.random.randn(input_size, hidden_size) * 0.01
b1 = np.zeros((1, hidden_size))
W2 = np.random.randn(hidden_size, output_size) * 0.01
b2 = np.zeros((1, output_size))

# Activation Selection
ACTIVATION = "softmax"

def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

# Forward Activation
def activate(z):
    if ACTIVATION == "relu":
        return relu(z)
    elif ACTIVATION == "sigmoid":
        return sigmoid(z)
    elif ACTIVATION == "tanh":
        return np.tanh(z)
    elif ACTIVATION == "linear":
        return z
    elif ACTIVATION == "softmax":
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)


# Backward Activation

def activate_derivative(z):
    if ACTIVATION == "relu":
        return relu_derivative(z)
    elif ACTIVATION == "sigmoid":
        return sigmoid_derivative(z)
    elif ACTIVATION == "tanh":
        return 1 - np.tanh(z) ** 2
    elif ACTIVATION == "linear":
        return 1
    elif ACTIVATION == "softmax":
        return 0

# Loss (MSE)
def loss(y, y_hat):
    return np.mean((y - y_hat) ** 2)

# Training with Backtracking
alpha_init = 1.0
beta = 0.8
c = 1e-4
epochs = 1000
m = len(X_train)

for _ in range(epochs):
    Z1 = X_train @ W1 + b1
    A1 = activate(Z1)
    y_hat = A1 @ W2 + b2
    current_loss = loss(y_train, y_hat)

    dy = (2 / m) * (y_hat - y_train)
    dW2 = A1.T @ dy
    db2 = np.sum(dy, axis=0, keepdims=True)

    dA1 = dy @ W2.T
    dZ1 = dA1 * activate_derivative(Z1)
    dW1 = X_train.T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)

    alpha = alpha_init

    while True:
        W1_new = W1 - alpha * dW1
        b1_new = b1 - alpha * db1
        W2_new = W2 - alpha * dW2
        b2_new = b2 - alpha * db2

        Z1_new = X_train @ W1_new + b1_new
        A1_new = activate(Z1_new)
        y_hat_new = A1_new @ W2_new + b2_new

        if loss(y_train, y_hat_new) <= current_loss - c * alpha * (
            np.sum(dW1**2) + np.sum(dW2**2)
        ):
            break

        alpha *= beta

    W1, b1, W2, b2 = W1_new, b1_new, W2_new, b2_new

# Testing
Z1_test = X_test @ W1 + b1
A1_test = activate(Z1_test)
y_pred = A1_test @ W2 + b2

# Evaluation
mae = np.mean(np.abs(y_test - y_pred))
print(f"Activation Used: {ACTIVATION}")
print("Mean Absolute Error (Rings):", round(mae, 3))

# Age Prediction
predicted_age = y_pred + 1.5
print("Sample Predicted Ages (years):")
print(predicted_age[:5].flatten())

Activation Used: softmax
Mean Absolute Error (Rings): 2.011
Sample Predicted Ages (years):
[11.49262948 11.44641324 11.46595781 11.47754116 11.5034415 ]
